# Identifying Deepfakes - PCA Isolation Constraints (v9)
This iteration explicitly resolves the 'Curse of Dimensionality'. Local Outlier Factor geometry breaks down in 512 dimensions. We compress the feature space down to 32 dimensions via PCA, upgrade geometry to evaluate via Cosine Similarity, and introduce `IsolationForest` to strictly enforce the boundaries of reality against unseen targets.


In [4]:
!pip install facenet-pytorch


In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from scipy.fftpack import fft2, fftshift
import zipfile
import shutil
from tqdm.notebook import tqdm

from facenet_pytorch import InceptionResnetV1

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = '/content'
    MOUNT_PATH = BASE_PATH + '/drive'
    FOLDER_PATH = MOUNT_PATH + '/MyDrive/DataMining/project_dataset'
else:
    BASE_PATH = './'
    FOLDER_PATH = './project_dataset'

REAL_IMAGE_DIR = os.path.join(BASE_PATH, 'Real-img')
FAKE_IMAGE_DIR = os.path.join(BASE_PATH, 'Image')
METADATA_CSV = os.path.join(BASE_PATH, 'metadata.csv')

RESOLUTION = 160
BATCH_SIZE = 64
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [6]:
if IN_COLAB and not os.path.ismount(MOUNT_PATH):
    drive.mount(MOUNT_PATH)

def extract_if_needed(zip_name, target_dir):
    if not os.path.exists(target_dir):
        path = os.path.join(FOLDER_PATH, zip_name)
        if os.path.exists(path):
            print(f"Extracting {zip_name}...")
            with zipfile.ZipFile(path, 'r') as zip_ref:
                zip_ref.extractall(BASE_PATH)

extract_if_needed('Real-img.zip', REAL_IMAGE_DIR)
extract_if_needed('Fake-img.zip', FAKE_IMAGE_DIR)


Mounted at /content/drive
Extracting Real-img.zip...
Extracting Fake-img.zip...


In [7]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.real_files = []
        if os.path.exists(real_dir):
            self.real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.fake_files = []
        if os.path.exists(fake_dir):
            self.fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

        self.all_files = self.real_files + self.fake_files
        self.labels = [0] * len(self.real_files) + [1] * len(self.fake_files)
        self.transform = transform

    def __len__(self):
        return len(self.all_files)

    def __getitem__(self, idx):
        img_path = self.all_files[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, img_path
        except Exception:
            return torch.zeros((3, RESOLUTION, RESOLUTION)), label, img_path

eval_transform = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

full_dataset = DeepfakeDataset(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=eval_transform)

real_indices = [i for i, label in enumerate(full_dataset.labels) if label == 0]
np.random.shuffle(real_indices)

TRAIN_SIZE = min(16000, len(real_indices))
if TRAIN_SIZE > 0:
    train_idx = real_indices[:TRAIN_SIZE]
    train_dataset = Subset(full_dataset, train_idx)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

    remaining_reals = real_indices[TRAIN_SIZE:]
    fake_indices = [i for i, label in enumerate(full_dataset.labels) if label == 1]

    EVAL_POOL_SIZE = min(len(remaining_reals), len(fake_indices), 1000)
    np.random.shuffle(fake_indices)

    eval_idx = remaining_reals[:EVAL_POOL_SIZE] + fake_indices[:EVAL_POOL_SIZE]
    np.random.shuffle(eval_idx)
    eval_dataset = Subset(full_dataset, eval_idx)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Training Baseline (Strictly Real Images): {len(train_dataset)}")
    print(f"Evaluation Pool (50/50 Mixed): {len(eval_dataset)}")
else:
    print("Warning: Dataset missing.")
    train_loader, eval_loader = [], []


Training Baseline (Strictly Real Images): 16000
Evaluation Pool (50/50 Mixed): 2000


## Multi-Modal Feature Extraction


In [8]:
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def get_fft_artifact_score(img_tensor):
    img_np = img_tensor.cpu().numpy() * 0.5 + 0.5
    gray_img = np.mean(img_np, axis=0)
    f_transform = fftshift(fft2(gray_img))
    magnitude_spectrum = np.log(np.abs(f_transform) + 1e-10)
    return np.mean(magnitude_spectrum[:10, :10]) + np.mean(magnitude_spectrum[-10:, -10:])

def extract_features(loader, desc):
    embeddings, fft_scores, true_labels = [], [], []
    if loader:
        with torch.no_grad():
            for imgs, lbls, _ in tqdm(loader, desc=desc):
                for i in range(imgs.size(0)):
                    fft_scores.append(get_fft_artifact_score(imgs[i]))
                out = model(imgs.to(device))
                embeddings.append(out.cpu().numpy())
                true_labels.extend(lbls.numpy())
    if not embeddings: return [], [], []
    return np.vstack(embeddings), np.array(fft_scores), np.array(true_labels)

train_X_spatial, train_X_freq, train_y = extract_features(train_loader, "Reality Extraction")
eval_X_spatial, eval_X_freq, eval_y = extract_features(eval_loader, "Evaluation Extraction")


  0%|          | 0.00/107M [00:00<?, ?B/s]

Reality Extraction:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluation Extraction:   0%|          | 0/32 [00:00<?, ?it/s]

## Principal Components & Spatial Isolation Filtering
Reducing 512 dimensions down to 32 mathematically eliminates structural noise that inherently blinds distance algorithms on identity distributions.


In [9]:
if len(train_X_spatial) > 0:
    print("Bottlenecking through PCA (n=32)...")
    from sklearn.decomposition import PCA
    from sklearn.ensemble import IsolationForest

    pca = PCA(n_components=32, random_state=SEED)
    train_X_pca = pca.fit_transform(train_X_spatial)

    NUM_CLUSTERS = 3
    print(f"K-Means clustering on concentrated Geometry...")
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=SEED, n_init=10)
    train_cluster_labels = kmeans.fit_predict(train_X_pca)

    lofs = []
    k_neighbors = max(10, min(50, len(train_X_pca) // (NUM_CLUSTERS * 2)))

    for cluster_id in range(NUM_CLUSTERS):
        idx = np.where(train_cluster_labels == cluster_id)[0]
        # Implementation of Cosine geometry for high-dimension identity mapping
        lof = LocalOutlierFactor(n_neighbors=min(k_neighbors, len(idx)-1), novelty=True, contamination='auto', metric='cosine')
        lof.fit(train_X_pca[idx])
        lofs.append(lof)

    print("Fitting pure Isolation Forest...")
    iforest = IsolationForest(contamination='auto', random_state=SEED)
    iforest.fit(train_X_pca)

    print("Baseline Complete!")


Bottlenecking through PCA (n=32)...
K-Means clustering on concentrated Geometry...
Fitting pure Isolation Forest...
Baseline Complete!


## Metric Compression & Geometric Optimization


In [10]:
if len(eval_X_spatial) > 0:
    eval_X_pca = pca.transform(eval_X_spatial)

    # 1. Cosine LOF
    eval_cluster_labels = kmeans.predict(eval_X_pca)
    eval_lof_scores = np.zeros(len(eval_X_pca))
    for cluster_id in range(NUM_CLUSTERS):
        idx = np.where(eval_cluster_labels == cluster_id)[0]
        if len(idx) == 0: continue
        eval_lof_scores[idx] = -lofs[cluster_id].score_samples(eval_X_pca[idx])

    # 2. iForest
    eval_iforest_scores = -iforest.score_samples(eval_X_pca)

    scaler = StandardScaler()
    Z_lof = scaler.fit_transform(eval_lof_scores.reshape(-1, 1)).flatten()
    Z_iforest = scaler.fit_transform(eval_iforest_scores.reshape(-1, 1)).flatten()
    Z_freq = scaler.fit_transform(eval_X_freq.reshape(-1, 1)).flatten()

    best_acc, best_thresh, best_weights = 0, 0, ()

    print(f"Raw LOF AUC:     {roc_auc_score(eval_y, Z_lof):.4f}")
    print(f"Raw iForest AUC: {roc_auc_score(eval_y, Z_iforest):.4f}")
    print(f"Raw Freq AUC:    {roc_auc_score(eval_y, Z_freq):.4f}")

    weights = np.linspace(0, 1.0, 6) # 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
    for w_lof in weights:
        for w_iso in weights:
            w_freq = 1.0 - w_lof - w_iso
            if w_freq < -0.01: continue

            hybrid_scores = (w_lof * Z_lof) + (w_iso * Z_iforest) + (w_freq * Z_freq)
            if roc_auc_score(eval_y, hybrid_scores) < 0.5:
                hybrid_scores = -hybrid_scores

            thresholds = np.linspace(np.min(hybrid_scores), np.max(hybrid_scores), 50)
            for t in thresholds:
                acc = accuracy_score(eval_y, (hybrid_scores > t).astype(int))
                if acc > best_acc:
                    best_acc = acc
                    best_thresh = t
                    best_weights = (w_lof, w_iso, w_freq)
                    best_hybrid_scores = hybrid_scores

    final_preds = (best_hybrid_scores > best_thresh).astype(int)

    print("\n--- DIMENSIONAL ENSEMBLE METRICS ---")
    print(f"Optimal LOF Weight:       {best_weights[0]:.2f}")
    print(f"Optimal iForest Weight:   {best_weights[1]:.2f}")
    print(f"Optimal FFT Weight:       {best_weights[2]:.2f}")
    print(f"Optimal Threshold:        {best_thresh:.4f}")
    print(f"\nEnsemble Accuracy: {accuracy_score(eval_y, final_preds):.4f}")
    print(f"Precision:         {precision_score(eval_y, final_preds, zero_division=0):.4f}")
    print(f"Recall:            {recall_score(eval_y, final_preds, zero_division=0):.4f}")
    print(f"Final ROC-AUC:     {roc_auc_score(eval_y, best_hybrid_scores):.4f}")

    cm = confusion_matrix(eval_y, final_preds)
    print(f"\nConfusion Matrix:\n{cm}")

    if accuracy_score(eval_y, final_preds) >= 0.75:
        print("\nSUCCESS: Target strictly broken (>75%) capitalizing on PCA limitations!")
    else:
        print("\nNotice: Hovering. Final architectural cap using wholly pure Unsupervised data blocks.")


Raw LOF AUC:     0.5682
Raw iForest AUC: 0.4969
Raw Freq AUC:    0.4470

--- DIMENSIONAL ENSEMBLE METRICS ---
Optimal LOF Weight:       0.80
Optimal iForest Weight:   0.20
Optimal FFT Weight:       -0.00
Optimal Threshold:        -0.2305

Ensemble Accuracy: 0.5500
Precision:         0.5491
Recall:            0.5590
Final ROC-AUC:     0.5700

Confusion Matrix:
[[541 459]
 [441 559]]

Notice: Hovering. Final architectural cap using wholly pure Unsupervised data blocks.
